<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/Blend_FIBA_DermaMNIST_PathMNIST_QSentry_Extension_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Blend/FIBA QSentry-style Extension — DermaMNIST and PathMNIST

This notebook contains **only Attack 2 Blend** and **Attack 3 FIBA-style** poisonous-image workflows. Blend runs first by default. FIBA is disabled until Blend is validated. No success is claimed unless CA/ASR and detection metrics are generated by the notebook.

## Claim guard

This notebook uses **QSentry-style / CC-QMC-inspired** measurement clustering and QMRS-probe features. It does not claim exact QSentry reproduction. Detection is blocked if ASR < 0.80 and the output is saved as `blocked_by_weak_attack`.

In [1]:

# ============================================================
# Imports, configuration, reproducibility
# Notebook 2: Blend / DermaMNIST first; FIBA / PathMNIST disabled by default
# ============================================================

import os, sys, math, json, time, random, warnings, subprocess
from dataclasses import dataclass, asdict, replace
from typing import Optional, Tuple, Dict, Any, List
warnings.filterwarnings("ignore")

KAGGLE_T4_MODE = True
INSTALL_MISSING_PACKAGES = True

def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError as exc:
        if not INSTALL_MISSING_PACKAGES:
            raise ImportError(f"Missing package '{import_name}'. Install it first. Original error: {exc}")
        pip_args = pip_name or import_name
        if isinstance(pip_args, str):
            pip_args = [pip_args]
        print(f"[install] Missing {import_name}; installing: {' '.join(pip_args)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pip_args])
        return __import__(import_name)

np = ensure_package("numpy")
pd = ensure_package("pandas")
torch = ensure_package("torch")
torchvision = ensure_package("torchvision")
sklearn = ensure_package("sklearn", "scikit-learn")
medmnist = ensure_package("medmnist")
qml = ensure_package("pennylane", ["pennylane", "pennylane-lightning"])
matplotlib = ensure_package("matplotlib")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset
import torchvision.transforms as T
from torchvision import models
from torchvision.models import ResNet18_Weights
from medmnist import INFO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA, PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score, classification_report,
    roc_curve, precision_recall_curve, auc, silhouette_score
)
import matplotlib.pyplot as plt
from IPython.display import display

DEFAULT_OUT_DIR = (
    "/kaggle/working/blend_fiba_qsentry_extension_outputs"
    if os.path.exists("/kaggle/working")
    else "/mnt/data/blend_fiba_qsentry_extension_outputs"
    if os.path.exists("/mnt/data")
    else "./blend_fiba_qsentry_extension_outputs"
)

# Default run policy: run Blend first. FIBA is disabled until Blend is validated.
RUN_DERMAMNIST_BLEND = True
RUN_PATHMNIST_FIBA = False
RUN_CLASSICAL_POISONED_BASELINE = True
RUN_PAIR_SELECTION = True
RUN_SANITIZATION = False  # enable only after detector passes ASR and detection gates

CANDIDATE_PAIRS = {
    "dermamnist_blend": [(0, 4), (1, 4), (2, 4), (5, 4), (6, 4)],
    "pathmnist_fiba": [(0, 5), (1, 5), (2, 5), (3, 5), (8, 5), (4, 7)],
}

@dataclass
class QMedShieldConfig:
    primary_seed: int = 42
    seeds: Tuple[int, ...] = (42, 123, 777)

    dataset_name: str = "dermamnist"
    attack_type: str = "blend"
    n_classes: int = 7
    native_image_size: int = 28
    model_image_size: int = 96
    source_class: Optional[int] = None
    target_class: Optional[int] = None

    max_train_n: int = 5000
    max_clean_test_n: int = 1200
    max_asr_n: int = 800
    val_size: float = 0.15

    use_imagenet_weights: bool = True
    freeze_resnet_lower_blocks: bool = True
    n_qubits: int = 8
    vqc_layers: int = 4
    obs_mode: str = "ZX_ALT"
    trojan_init_std: float = 0.0  # no QTrojan here; VQC still exists for QNN measurement features

    train_batch_size: int = 48
    eval_batch_size: int = 96
    clean_epochs: int = 8
    qtrojan_epochs: int = 0
    lr_base: float = 5e-4
    lr_trojan_mult: float = 1.0
    weight_decay: float = 1e-4
    label_smoothing: float = 0.10
    clean_acc_min: float = 0.50
    asr_min: float = 0.80
    strict_attack_gate: bool = True
    freeze_classical_in_trojan_stage: bool = False
    clean_loss_weight_trojan: float = 0.0
    q_loss_weight_trojan: float = 0.0
    qtrojan_source_boost_batches: int = 0

    # Detector config
    ica_components: int = 4
    k_clusters: int = 3
    kmeans_n_init: int = 10
    fpr_target: float = 0.05
    k_values: Tuple[int, ...] = (3, 5, 8, 10)
    threshold_method: str = "clean_val_95pct"  # or top_expected_poison_count

    # QSentry-style pool counts
    clean_source_n: int = 450
    clean_target_n: int = 500
    poison_n: int = 50

    # Blend
    blend_alpha: float = 0.20
    blend_alpha_values: Tuple[float, ...] = (0.20, 0.15, 0.10, 0.07)
    blend_poison_rate: float = 0.15

    # FIBA-style
    fiba_strength: float = 0.25
    fiba_strength_values: Tuple[float, ...] = (0.35, 0.25, 0.18)
    fiba_poison_rate: float = 0.15
    fiba_low_freq_radius: int = 10

    # Pilot selection
    pilot_max_train_n: int = 1500
    pilot_epochs: int = 2

    # Output
    out_dir: str = DEFAULT_OUT_DIR

cfg = QMedShieldConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NEEDS_VERIFICATION = []
PLANNED_PENDING = []

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(cfg.primary_seed)
print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Output directory:", cfg.out_dir)
display(pd.DataFrame([asdict(cfg)]).T.rename(columns={0: "value"}))


[install] Missing medmnist; installing: medmnist
[install] Missing pennylane; installing: pennylane pennylane-lightning
Device: cpu
Torch: 2.11.0+cpu
PennyLane: 0.45.0
Output directory: ./blend_fiba_qsentry_extension_outputs


,value
primary_seed,42
seeds,"(42, 123, 777)"
dataset_name,dermamnist
attack_type,blend
n_classes,7
native_image_size,28
model_image_size,96
source_class,None
target_class,None
max_train_n,5000


In [2]:

# ============================================================
# Dataset loading utilities for DermaMNIST and PathMNIST
# ============================================================

def load_medmnist_split(dataset_name: str, split: str, download: bool = True):
    if dataset_name not in INFO:
        raise ValueError(f"Unknown MedMNIST dataset: {dataset_name}")
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info["python_class"])
    ds = DataClass(split=split, download=download)
    imgs = np.asarray(ds.imgs)
    labels = np.asarray(ds.labels).reshape(-1).astype(np.int64)
    if imgs.ndim == 3:
        imgs = imgs[..., None]
    if imgs.shape[-1] == 1:
        imgs = np.repeat(imgs, 3, axis=-1)
    if imgs.shape[-1] != 3:
        raise ValueError(f"Expected RGB/grayscale data, got {imgs.shape}")
    X = torch.tensor(imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    y = torch.tensor(labels, dtype=torch.long)
    return X, y, info

def label_map_to_int(label_map):
    out = {}
    for k, v in label_map.items():
        try:
            out[int(k)] = str(v)
        except Exception:
            pass
    return dict(sorted(out.items()))

def load_dataset_bundle(dataset_name: str):
    X_train, y_train, info = load_medmnist_split(dataset_name, "train", download=True)
    X_val, y_val, _ = load_medmnist_split(dataset_name, "val", download=True)
    X_test, y_test, _ = load_medmnist_split(dataset_name, "test", download=True)
    label_map = label_map_to_int(info.get("label", {}))
    print(f"Loaded {dataset_name}:", X_train.shape, X_val.shape, X_test.shape)
    display(pd.DataFrame({"class_id": list(label_map.keys()), "class_name": list(label_map.values())}))
    return {
        "dataset_name": dataset_name,
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "label_map": label_map,
        "n_classes": len(label_map),
    }

def resize_tensor_images(X, size):
    return F.interpolate(X, size=(size, size), mode="bilinear", align_corners=False).clamp(0, 1)

def clone_cfg(base_cfg, **updates):
    data = asdict(base_cfg)
    data.update(updates)
    return QMedShieldConfig(**data)


In [3]:
# ============================================================
# S3-S7. MODEL + TRAINING HELPERS
# ============================================================

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

def preprocess_for_resnet(x, model_image_size):
    x = F.interpolate(x, size=(model_image_size, model_image_size), mode="bilinear", align_corners=False)
    return (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)

def measurement_dim(n_qubits, obs_mode):
    obs = obs_mode.upper().strip()
    if obs in ["Z_ONLY", "ZX_ALT"]:
        return n_qubits
    if obs == "XYZ":
        return 3 * n_qubits
    raise ValueError(f"Unknown obs_mode={obs_mode}")

def make_vqc_qnode(n_qubits, n_layers, obs_mode="ZX_ALT"):
    try:
        dev = qml.device("lightning.qubit", wires=n_qubits)
    except Exception:
        print("[warning] lightning.qubit unavailable; using default.qubit")
        dev = qml.device("default.qubit", wires=n_qubits)
    obs = obs_mode.upper().strip()

    @qml.qnode(dev, interface="torch", diff_method="best")
    def circuit(angles, weights, input_scales, input_bias, trojan_angles, trigger_flag):
        for layer in range(n_layers):
            for q in range(n_qubits):
                qml.Rot(weights[layer, q, 0], weights[layer, q, 1], weights[layer, q, 2], wires=q)
            for q in range(n_qubits):
                qml.RY(input_scales[layer, q] * angles[q] + input_bias[layer, q], wires=q)

            # Critical isolation: trigger_flag=0 disables the inserted QTrojan RY gate.
            for q in range(n_qubits):
                qml.RY(trigger_flag * trojan_angles[layer, q], wires=q)

            if layer % 2 == 0:
                for q in range(n_qubits):
                    qml.CNOT(wires=[q, (q + 1) % n_qubits])
            else:
                for q in range(n_qubits):
                    qml.CNOT(wires=[(q + 1) % n_qubits, q])

        if obs == "Z_ONLY":
            return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]
        if obs == "XYZ":
            out = []
            for q in range(n_qubits):
                out.extend([qml.expval(qml.PauliX(q)), qml.expval(qml.PauliY(q)), qml.expval(qml.PauliZ(q))])
            return out
        return [qml.expval(qml.PauliZ(q)) if q % 2 == 0 else qml.expval(qml.PauliX(q)) for q in range(n_qubits)]
    return circuit

class StrongVQCMeasurementLayer(nn.Module):
    def __init__(self, in_dim, n_qubits, n_layers, obs_mode, trojan_init_std):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.obs_mode = obs_mode
        self.compress = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits),
            nn.Tanh(),
        )
        self.weights = nn.Parameter(0.01 * torch.randn(n_layers, n_qubits, 3))
        self.input_scales = nn.Parameter(torch.ones(n_layers, n_qubits))
        self.input_bias = nn.Parameter(torch.zeros(n_layers, n_qubits))
        self.trojan_angles = nn.Parameter(trojan_init_std * torch.randn(n_layers, n_qubits))
        self.qnode = make_vqc_qnode(n_qubits, n_layers, obs_mode)

    def forward(self, context, trojan_mask=None):
        angles = math.pi * (self.compress(context) + 1.0) / 2.0
        if trojan_mask is None:
            trojan_mask = torch.zeros(angles.shape[0], device=angles.device, dtype=angles.dtype)
        else:
            trojan_mask = trojan_mask.to(device=angles.device, dtype=angles.dtype).view(-1)

        vals = [
            torch.stack(self.qnode(a, self.weights, self.input_scales, self.input_bias, self.trojan_angles, flag)).float()
            for a, flag in zip(angles, trojan_mask)
        ]
        return torch.stack(vals, dim=0)

class QMedShieldHybridQNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        weights = None
        if cfg.use_imagenet_weights:
            try:
                weights = ResNet18_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            resnet = models.resnet18(weights=weights)
        except Exception as exc:
            msg = f"needs verification: ImageNet weights unavailable ({exc}); using random ResNet-18 init."
            print("[warning]", msg)
            NEEDS_VERIFICATION.append(msg)
            resnet = models.resnet18(weights=None)

        self.stem = nn.Sequential(*list(resnet.children())[:-2])
        if cfg.freeze_resnet_lower_blocks:
            for p in self.stem[:6].parameters():
                p.requires_grad = False

        self.rnn = nn.GRU(input_size=512, hidden_size=256, num_layers=2,
                          batch_first=True, bidirectional=True, dropout=0.2)
        self.vqc = StrongVQCMeasurementLayer(512, cfg.n_qubits, cfg.vqc_layers, cfg.obs_mode, cfg.trojan_init_std)
        self.q_head = nn.Linear(measurement_dim(cfg.n_qubits, cfg.obs_mode), cfg.n_classes)

    def _context(self, x_preprocessed):
        feats = self.stem(x_preprocessed)
        B, C, H, W = feats.shape
        seq = feats.view(B, C, H*W).permute(0, 2, 1)
        out, _ = self.rnn(seq)
        return out.mean(dim=1)

    def _mask(self, batch_size, device, t=0.0, trojan_mask=None):
        if trojan_mask is not None:
            return trojan_mask.to(device=device, dtype=torch.float32).view(-1)
        if isinstance(t, torch.Tensor):
            return t.to(device=device, dtype=torch.float32).view(-1)
        return torch.full((batch_size,), float(t), device=device, dtype=torch.float32)

    def forward(self, x_preprocessed, t=0.0, return_measurements=False, trojan_mask=None):
        ctx = self._context(x_preprocessed)
        mask = self._mask(ctx.shape[0], ctx.device, t=t, trojan_mask=trojan_mask)
        qfeat = self.vqc(ctx, trojan_mask=mask)
        logits = self.q_head(qfeat)
        if return_measurements:
            return logits, qfeat, ctx
        return logits

    @torch.no_grad()
    def extract_vqc_measurements(self, x_raw, t=0.0):
        xp = preprocess_for_resnet(x_raw, self.cfg.model_image_size)
        _, qfeat, _ = self.forward(xp, t=t, return_measurements=True)
        return qfeat

def verify_architecture(model, cfg, device):
    model = model.to(device)
    dummy = torch.rand(4, 3, cfg.native_image_size, cfg.native_image_size, device=device)
    xp = preprocess_for_resnet(dummy, cfg.model_image_size)
    logits, qfeat, ctx = model(xp, t=0, return_measurements=True)
    assert logits.shape == (4, cfg.n_classes), logits.shape
    assert qfeat.shape == (4, measurement_dim(cfg.n_qubits, cfg.obs_mode)), qfeat.shape
    assert ctx.shape == (4, 512), ctx.shape
    print("Architecture verified:", logits.shape, qfeat.shape, ctx.shape)

def stratified_limit_tensors(X, y, max_n, seed):
    if max_n is None or len(y) <= max_n:
        return X, y, np.arange(len(y))
    idx = np.arange(len(y))
    _, keep = train_test_split(idx, test_size=max_n, random_state=seed, stratify=y.cpu().numpy())
    keep = np.sort(keep)
    return X[keep], y[keep], keep

def make_train_val_split(X, y, cfg, seed):
    X_lim, y_lim, keep = stratified_limit_tensors(X, y, cfg.max_train_n, seed)
    idx = np.arange(len(y_lim))
    tr, va = train_test_split(idx, test_size=cfg.val_size, random_state=seed, stratify=y_lim.cpu().numpy())
    tr, va = np.sort(tr), np.sort(va)
    return {"X_train": X_lim[tr], "y_train": y_lim[tr], "X_val": X_lim[va], "y_val": y_lim[va], "source_idx": keep}

def augment_train_only(X, seed):
    set_all_seeds(seed)
    aug = T.Compose([T.RandomHorizontalFlip(p=0.5), T.RandomCrop(X.shape[-1], padding=2, padding_mode="reflect")])
    return torch.stack([aug(img) for img in X])

def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TensorDataset(X.float(), y.long()), batch_size=batch_size, shuffle=shuffle, num_workers=0)

def set_trainable(module, trainable):
    for p in module.parameters():
        p.requires_grad = trainable

def optimizer_for_stage(model, cfg, stage):
    if stage == "trojan":
        params = []
        params.append({"params": [model.vqc.trojan_angles], "lr": cfg.lr_base * cfg.lr_trojan_mult})
        other_vqc = [p for n, p in model.vqc.named_parameters() if n != "trojan_angles" and p.requires_grad]
        if other_vqc:
            params.append({"params": other_vqc, "lr": cfg.lr_base})
        params.append({"params": model.q_head.parameters(), "lr": cfg.lr_base})
        return torch.optim.AdamW(params, weight_decay=cfg.weight_decay)

    return torch.optim.AdamW([
        {"params": model.stem[6:].parameters(), "lr": cfg.lr_base * 0.1},
        {"params": model.rnn.parameters(), "lr": cfg.lr_base},
        {"params": model.vqc.parameters(), "lr": cfg.lr_base},
        {"params": model.q_head.parameters(), "lr": cfg.lr_base},
    ], weight_decay=cfg.weight_decay)

@torch.no_grad()
def evaluate_classifier(model, X, y, cfg, device, trojan_t=0.0):
    model.eval()
    loader = make_loader(X, y, cfg.eval_batch_size, shuffle=False)
    preds, probs = [], []
    for xb, _ in loader:
        xb = preprocess_for_resnet(xb.to(device), cfg.model_image_size)
        logits = model(xb, t=trojan_t)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        preds.append(prob.argmax(axis=1))
        probs.append(prob)
    preds = np.concatenate(preds)
    probs = np.concatenate(probs)
    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
    return {
        "accuracy": float(accuracy_score(y_np, preds)),
        "macro_f1": float(f1_score(y_np, preds, average="macro", zero_division=0)),
        "preds": preds,
        "probs": probs,
    }

@torch.no_grad()
def evaluate_qtrojan_asr(model, X, y, cfg, device, seed):
    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
    idx = np.where(y_np == cfg.source_class)[0]
    if len(idx) == 0:
        raise ValueError("No source-class samples for ASR.")
    rng = np.random.default_rng(seed)
    if cfg.max_asr_n and len(idx) > cfg.max_asr_n:
        idx = rng.choice(idx, size=cfg.max_asr_n, replace=False)
    idx = np.sort(idx)
    y_target = torch.full((len(idx),), cfg.target_class, dtype=torch.long)
    out = evaluate_classifier(model, X[idx], y_target, cfg, device, trojan_t=1.0)
    out["n_source_eval"] = int(len(idx))
    return out

def train_clean_stage(model, train_loader, val_loader, cfg, device):
    model = model.to(device)
    opt = optimizer_for_stage(model, cfg, "clean")
    hist = []
    print(f"Stage A clean training: {cfg.clean_epochs} epochs")
    for ep in range(1, cfg.clean_epochs + 1):
        model.train()
        loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xb = preprocess_for_resnet(xb.to(device), cfg.model_image_size)
            yb = yb.to(device)
            opt.zero_grad()
            logits = model(xb, t=0.0)
            loss = F.cross_entropy(logits, yb, label_smoothing=cfg.label_smoothing)
            loss.backward()
            opt.step()
            loss_sum += float(loss.detach().cpu()) * len(xb)
            n += len(xb)
        val = evaluate_classifier(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], cfg, device, 0.0)
        row = {"epoch": ep, "stage": "clean", "loss": loss_sum/max(n,1), "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]}
        hist.append(row)
        print(f"Epoch {ep:02d}: loss={row['loss']:.4f}, val_CA={row['val_acc']:.4f}")
    return model, hist

def train_qtrojan_stage(model, train_loader, val_loader, cfg, device):
    """
    QTrojan optimization stage.

    The previous short run underfit the attack (ASR below the configured gate).
    This version keeps the classical encoder frozen by default, but adds a small
    source-only trigger boost so every epoch sees enough source->target attack
    signal without making the notebook too heavy for Kaggle T4.
    """
    model = model.to(device)
    if cfg.freeze_classical_in_trojan_stage:
        set_trainable(model.stem, False)
        set_trainable(model.rnn, False)
        print("Frozen ResNet/BiGRU during QTrojan stage.")
    set_trainable(model.vqc, True)
    set_trainable(model.q_head, True)

    # Source-only loader for stable trigger optimization.
    X_all, y_all = train_loader.dataset.tensors
    src_mask = (y_all == cfg.source_class)
    if int(src_mask.sum()) == 0:
        raise ValueError("No source-class samples in training split; cannot train QTrojan.")
    src_dataset = TensorDataset(X_all[src_mask], y_all[src_mask])
    src_loader = DataLoader(
        src_dataset,
        batch_size=min(cfg.train_batch_size, len(src_dataset)),
        shuffle=True,
        drop_last=False
    )

    opt = optimizer_for_stage(model, cfg, "trojan")
    hist = []
    print(f"Stage B QTrojan training: {cfg.qtrojan_epochs} epochs")
    for ep in range(1, cfg.qtrojan_epochs + 1):
        model.train()
        loss_sum, n, tbatches = 0.0, 0, 0

        # Mixed clean + in-batch source attack pass.
        for xb, yb in train_loader:
            xb = preprocess_for_resnet(xb.to(device), cfg.model_image_size)
            yb = yb.to(device)
            opt.zero_grad()

            logits_clean = model(xb, t=0.0)
            loss_clean = F.cross_entropy(logits_clean, yb, label_smoothing=cfg.label_smoothing)

            src_idx = torch.where(yb == cfg.source_class)[0]
            if len(src_idx) > 0:
                target = torch.full((len(src_idx),), cfg.target_class, dtype=torch.long, device=device)
                logits_trig = model(xb[src_idx], t=1.0)
                loss_trig = F.cross_entropy(logits_trig, target)
                loss = cfg.clean_loss_weight_trojan * loss_clean + cfg.q_loss_weight_trojan * loss_trig
                tbatches += 1
            else:
                loss = cfg.clean_loss_weight_trojan * loss_clean

            loss.backward()
            opt.step()
            loss_sum += float(loss.detach().cpu()) * len(xb)
            n += len(xb)

        # Lightweight source-only trigger boost. This usually improves ASR
        # without unfreezing the full ResNet/BiGRU stack.
        if cfg.qtrojan_source_boost_batches > 0:
            for b_i, (xs, _) in enumerate(src_loader):
                if b_i >= cfg.qtrojan_source_boost_batches:
                    break
                xs = preprocess_for_resnet(xs.to(device), cfg.model_image_size)
                target = torch.full((len(xs),), cfg.target_class, dtype=torch.long, device=device)
                opt.zero_grad()
                logits_trig = model(xs, t=1.0)
                loss_boost = cfg.q_loss_weight_trojan * F.cross_entropy(logits_trig, target)
                loss_boost.backward()
                opt.step()
                loss_sum += float(loss_boost.detach().cpu()) * len(xs)
                n += len(xs)
                tbatches += 1

        val_ca = evaluate_classifier(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], cfg, device, 0.0)
        val_asr = evaluate_qtrojan_asr(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], cfg, device, cfg.primary_seed)
        row = {"epoch": ep, "stage": "qtrojan", "loss": loss_sum/max(n,1), "val_ca": val_ca["accuracy"], "val_asr": val_asr["accuracy"], "trojan_batches": tbatches}
        hist.append(row)
        print(f"Epoch {ep:02d}: loss={row['loss']:.4f}, val_CA={row['val_ca']:.4f}, val_ASR={row['val_asr']:.4f}")
        if val_ca["accuracy"] >= cfg.clean_acc_min and val_asr["accuracy"] >= cfg.asr_min:
            print("Early stop: CA/ASR gates reached.")
            break
    return model, hist


In [4]:

# ============================================================
# Classical poisoned-model baseline, data-level attacks, QSentry-style / CC-QMC-inspired detection
# ============================================================

class ClassicalResNetBiGRU(nn.Module):
    """Classical CONTROL baseline only. Proposed model remains QMedShieldHybridQNN with PennyLane VQC."""
    def __init__(self, local_cfg):
        super().__init__()
        self.cfg = local_cfg
        weights = None
        if local_cfg.use_imagenet_weights:
            try:
                weights = ResNet18_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            resnet = models.resnet18(weights=weights)
        except Exception:
            resnet = models.resnet18(weights=None)
        self.stem = nn.Sequential(*list(resnet.children())[:-2])
        if local_cfg.freeze_resnet_lower_blocks:
            for p in self.stem[:6].parameters():
                p.requires_grad = False
        self.rnn = nn.GRU(input_size=512, hidden_size=256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.head = nn.Linear(512, local_cfg.n_classes)

    def context(self, x_preprocessed):
        feats = self.stem(x_preprocessed)
        B, C, H, W = feats.shape
        seq = feats.view(B, C, H * W).permute(0, 2, 1)
        out, _ = self.rnn(seq)
        return out.mean(dim=1)

    def forward(self, x_preprocessed, return_context=False):
        ctx = self.context(x_preprocessed)
        logits = self.head(ctx)
        if return_context:
            return logits, ctx
        return logits

@torch.no_grad()
def evaluate_classical_classifier(model, X, y, local_cfg, device):
    model.eval()
    loader = make_loader(X, y, local_cfg.eval_batch_size, shuffle=False)
    preds, probs = [], []
    for xb, _ in loader:
        xb = preprocess_for_resnet(xb.to(device), local_cfg.model_image_size)
        logits = model(xb)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        preds.append(prob.argmax(axis=1)); probs.append(prob)
    preds = np.concatenate(preds); probs = np.concatenate(probs)
    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
    return {"accuracy": float(accuracy_score(y_np, preds)), "macro_f1": float(f1_score(y_np, preds, average="macro", zero_division=0)), "preds": preds, "probs": probs}

def train_classical_stage(model, train_loader, val_loader, local_cfg, device, label="classical_poisoned"):
    model = model.to(device)
    opt = torch.optim.AdamW([
        {"params": model.stem[6:].parameters(), "lr": local_cfg.lr_base * 0.1},
        {"params": model.rnn.parameters(), "lr": local_cfg.lr_base},
        {"params": model.head.parameters(), "lr": local_cfg.lr_base},
    ], weight_decay=local_cfg.weight_decay)
    hist = []
    print(f"[{label}] training: {local_cfg.clean_epochs} epochs")
    for ep in range(1, local_cfg.clean_epochs + 1):
        model.train(); loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xb = preprocess_for_resnet(xb.to(device), local_cfg.model_image_size)
            yb = yb.to(device)
            opt.zero_grad(); logits = model(xb)
            loss = F.cross_entropy(logits, yb, label_smoothing=local_cfg.label_smoothing)
            loss.backward(); opt.step()
            loss_sum += float(loss.detach().cpu()) * len(xb); n += len(xb)
        val = evaluate_classical_classifier(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], local_cfg, device)
        hist.append({"epoch": ep, "stage": label, "loss": loss_sum/max(n,1), "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]})
        print(f"[{label}] Epoch {ep:02d}: loss={hist[-1]['loss']:.4f}, val_CA={hist[-1]['val_acc']:.4f}")
    return model, hist

# -----------------------------
# Attack implementations
# -----------------------------
def fixed_blend_trigger(shape, seed):
    rng = np.random.default_rng(seed)
    C, H, W = shape
    return torch.tensor(rng.random((C, H, W)), dtype=torch.float32)

def apply_blend(X, trigger, alpha):
    if trigger.dim() == 3:
        trigger = trigger.unsqueeze(0)
    return ((1 - alpha) * X + alpha * trigger.to(X.device)).clamp(0, 1)

def make_target_mean_trigger(X_train_96, y_train, target_class):
    idx = torch.where(y_train == target_class)[0]
    if len(idx) == 0:
        raise ValueError("No target-class samples for target mean trigger.")
    return X_train_96[idx].mean(dim=0, keepdim=True).clamp(0, 1)

def apply_fiba_lowfreq_batch(X, trigger, alpha=0.25, radius=10):
    """FIBA-style low-frequency amplitude injection at 96×96. Keeps input phase, injects trigger amplitude."""
    X_np = X.detach().cpu().numpy().astype(np.float32)
    trig = trigger.detach().cpu().numpy().astype(np.float32)
    if trig.ndim == 3:
        trig = trig[None]
    if trig.shape[0] == 1:
        trig = np.repeat(trig, X_np.shape[0], axis=0)
    out = np.zeros_like(X_np)
    for i in range(X_np.shape[0]):
        for c in range(X_np.shape[1]):
            Xf = np.fft.fft2(X_np[i, c])
            Tf = np.fft.fft2(trig[i, c])
            X_amp, X_phase = np.abs(Xf), np.angle(Xf)
            T_amp = np.abs(Tf)
            X_amp_shift = np.fft.fftshift(X_amp)
            T_amp_shift = np.fft.fftshift(T_amp)
            H, W = X_amp.shape; cy, cx = H // 2, W // 2
            yy, xx = np.ogrid[:H, :W]
            mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2
            mixed_shift = X_amp_shift.copy()
            mixed_shift[mask] = (1 - alpha) * X_amp_shift[mask] + alpha * T_amp_shift[mask]
            mixed_amp = np.fft.ifftshift(mixed_shift)
            out[i, c] = np.real(np.fft.ifft2(mixed_amp * np.exp(1j * X_phase)))
    return torch.tensor(np.clip(out, 0, 1), dtype=X.dtype)

def make_poisoned_train_tensors(X_train_96, y_train, local_cfg, attack_name, seed):
    y_np = y_train.cpu().numpy()
    src_idx = np.where(y_np == local_cfg.source_class)[0]
    if len(src_idx) == 0:
        raise ValueError("No source-class samples available for poisoning.")
    rng = np.random.default_rng(seed)
    poison_rate = local_cfg.blend_poison_rate if attack_name == "blend" else local_cfg.fiba_poison_rate
    n_poison = max(1, int(round(len(src_idx) * poison_rate)))
    n_poison = min(n_poison, len(src_idx))
    poison_idx = np.sort(rng.choice(src_idx, size=n_poison, replace=False))
    Xp, yp = X_train_96.clone(), y_train.clone()
    mask = torch.zeros(len(Xp), dtype=torch.bool)
    if attack_name == "blend":
        trigger = fixed_blend_trigger(tuple(Xp.shape[1:]), seed).unsqueeze(0)
        Xp[poison_idx] = apply_blend(Xp[poison_idx], trigger, local_cfg.blend_alpha)
    elif attack_name == "fiba":
        trigger = make_target_mean_trigger(X_train_96, y_train, local_cfg.target_class)
        Xp[poison_idx] = apply_fiba_lowfreq_batch(Xp[poison_idx], trigger, alpha=local_cfg.fiba_strength, radius=local_cfg.fiba_low_freq_radius)
    else:
        raise ValueError(f"Unknown attack: {attack_name}")
    yp[poison_idx] = local_cfg.target_class
    mask[poison_idx] = True
    return Xp, yp, mask, trigger

@torch.no_grad()
def evaluate_trigger_asr(model, X_test_96, y_test, trigger, local_cfg, device, attack_name, classical=False, seed=42):
    idx = np.where(y_test.cpu().numpy() == local_cfg.source_class)[0]
    if len(idx) == 0:
        raise ValueError("No source-class samples for ASR.")
    rng = np.random.default_rng(seed)
    if local_cfg.max_asr_n and len(idx) > local_cfg.max_asr_n:
        idx = rng.choice(idx, local_cfg.max_asr_n, replace=False)
    idx = np.sort(idx)
    X_src = X_test_96[idx]
    if attack_name == "blend":
        X_trig = apply_blend(X_src, trigger, local_cfg.blend_alpha)
    else:
        X_trig = apply_fiba_lowfreq_batch(X_src, trigger, alpha=local_cfg.fiba_strength, radius=local_cfg.fiba_low_freq_radius)
    y_target = torch.full((len(idx),), local_cfg.target_class, dtype=torch.long)
    if classical:
        return evaluate_classical_classifier(model, X_trig, y_target, local_cfg, device)
    return evaluate_classifier(model, X_trig, y_target, local_cfg, device, 0.0)

# -----------------------------
# Training one poisoned QNN / classical model
# -----------------------------
def make_split_from_bundle(bundle, local_cfg, seed, max_train_n=None):
    X_train_96 = resize_tensor_images(bundle["X_train"], local_cfg.model_image_size)
    X_lim, y_lim, _ = stratified_limit_tensors(X_train_96, bundle["y_train"], max_train_n or local_cfg.max_train_n, seed)
    idx = np.arange(len(y_lim))
    tr, va = train_test_split(idx, test_size=local_cfg.val_size, random_state=seed, stratify=y_lim.cpu().numpy())
    return {"X_train": X_lim[np.sort(tr)], "y_train": y_lim[np.sort(tr)], "X_val": X_lim[np.sort(va)], "y_val": y_lim[np.sort(va)]}

def train_poisoned_qnn_and_classical(
    bundle, local_cfg, attack_name, seed,
    max_train_n=None,
    epochs_override=None,
    eval_split="val",
    train_classical_baseline=False,
):
    """
    Train the poisoned QNN and, optionally, a matched classical poisoned-model baseline.

    Critical leakage rule:
      - Pilot pair/strength search MUST call eval_split="val" and train_classical_baseline=False.
      - Final frozen run MAY call eval_split="test" and train_classical_baseline=True.

    No test data is used during pilot pair/strength selection.
    """
    assert eval_split in ["val", "test"], "eval_split must be 'val' for pilot or 'test' for final run."
    split = make_split_from_bundle(bundle, local_cfg, seed, max_train_n=max_train_n)
    if epochs_override is not None:
        local_cfg = clone_cfg(local_cfg, clean_epochs=epochs_override)

    Xp, yp, poison_mask, trigger = make_poisoned_train_tensors(split["X_train"], split["y_train"], local_cfg, attack_name, seed)
    train_loader = make_loader(Xp, yp, local_cfg.train_batch_size, shuffle=True)
    val_loader = make_loader(split["X_val"], split["y_val"], local_cfg.eval_batch_size, shuffle=False)

    qnn = QMedShieldHybridQNN(local_cfg).to(DEVICE)
    verify_architecture(qnn, local_cfg, DEVICE)
    qnn, qnn_hist = train_clean_stage(qnn, train_loader, val_loader, local_cfg, DEVICE)

    classical, classical_hist = None, []
    if train_classical_baseline:
        classical = ClassicalResNetBiGRU(local_cfg).to(DEVICE)
        classical, classical_hist = train_classical_stage(
            classical, train_loader, val_loader, local_cfg, DEVICE,
            label=f"classical_{attack_name}_poisoned_final"
        )

    if eval_split == "val":
        X_eval_raw = split["X_val"]
        y_eval = split["y_val"]
    else:
        X_eval_raw = bundle["X_test"]
        y_eval = bundle["y_test"]

    X_eval_96 = resize_tensor_images(X_eval_raw, local_cfg.model_image_size)
    X_eval_lim, y_eval_lim, _ = stratified_limit_tensors(X_eval_96, y_eval, local_cfg.max_clean_test_n, seed)
    qnn_ca = evaluate_classifier(qnn, X_eval_lim, y_eval_lim, local_cfg, DEVICE, 0.0)
    qnn_asr = evaluate_trigger_asr(
        qnn, X_eval_96, y_eval, trigger, local_cfg, DEVICE,
        attack_name, classical=False, seed=seed
    )

    classical_ca, classical_asr = None, None
    if classical is not None:
        classical_ca = evaluate_classical_classifier(classical, X_eval_lim, y_eval_lim, local_cfg, DEVICE)
        classical_asr = evaluate_trigger_asr(
            classical, X_eval_96, y_eval, trigger, local_cfg, DEVICE,
            attack_name, classical=True, seed=seed
        )

    return {
        "qnn": qnn, "classical": classical, "trigger": trigger,
        "poison_mask": poison_mask, "split": split,
        "qnn_ca": qnn_ca, "qnn_asr": qnn_asr,
        "classical_ca": classical_ca, "classical_asr": classical_asr,
        "qnn_hist": qnn_hist, "classical_hist": classical_hist,
        "local_cfg": local_cfg, "attack_name": attack_name,
        "eval_split": eval_split,
        "classical_trained": bool(classical is not None),
    }

# -----------------------------
# QSentry-style / CC-QMC-inspired detection
# -----------------------------
def apply_attack_to_batch(X, trigger, local_cfg, attack_name):
    if attack_name == "blend":
        return apply_blend(X, trigger, local_cfg.blend_alpha)
    return apply_fiba_lowfreq_batch(X, trigger, alpha=local_cfg.fiba_strength, radius=local_cfg.fiba_low_freq_radius)

def build_qsentry_poison_pool(X, y, trigger, local_cfg, attack_name, seed, clean_source_n=None, clean_target_n=None, poison_n=None):
    X96 = resize_tensor_images(X, local_cfg.model_image_size) if X.shape[-1] != local_cfg.model_image_size else X
    y_np = y.cpu().numpy()
    src_idx = np.where(y_np == local_cfg.source_class)[0]
    tgt_idx = np.where(y_np == local_cfg.target_class)[0]
    rng = np.random.default_rng(seed)
    src_idx = rng.permutation(src_idx); tgt_idx = rng.permutation(tgt_idx)
    poison_n = poison_n or local_cfg.poison_n
    clean_source_n = clean_source_n or local_cfg.clean_source_n
    clean_target_n = clean_target_n or local_cfg.clean_target_n
    if len(src_idx) < clean_source_n + poison_n:
        poison_n = min(poison_n, max(1, len(src_idx)//10))
        clean_source_n = min(clean_source_n, max(1, len(src_idx)-poison_n))
    clean_target_n = min(clean_target_n, len(tgt_idx))
    if clean_source_n < 10 or clean_target_n < 10 or poison_n < 1:
        raise ValueError(f"Insufficient samples for QSentry-style pool: src={len(src_idx)}, tgt={len(tgt_idx)}")
    clean_src_idx = src_idx[:clean_source_n]
    poison_src_idx = src_idx[clean_source_n:clean_source_n+poison_n]
    clean_tgt_idx = tgt_idx[:clean_target_n]
    X_clean_src = X96[clean_src_idx]
    X_clean_tgt = X96[clean_tgt_idx]
    X_poison = apply_attack_to_batch(X96[poison_src_idx].clone(), trigger, local_cfg, attack_name)
    X_pool = torch.cat([X_clean_src, X_clean_tgt, X_poison], dim=0)
    y_poison = np.concatenate([np.zeros(len(X_clean_src)+len(X_clean_tgt), dtype=int), np.ones(len(X_poison), dtype=int)])
    groups = np.array(["clean_source"]*len(X_clean_src) + ["clean_target"]*len(X_clean_tgt) + ["poisonous_triggered_source"]*len(X_poison))
    summary = pd.DataFrame({"Group":["Clean source","Clean target","Poisonous triggered source","Total","Poison ratio"], "Count":[len(X_clean_src),len(X_clean_tgt),len(X_poison),len(X_pool), f"{100*len(X_poison)/len(X_pool):.2f}%"]})
    display(summary)
    return X_pool, y_poison, groups, summary

@torch.no_grad()
def extract_qnn_features(model, X, local_cfg, device):
    model.eval(); q_rows, ctx_rows, prob_rows = [], [], []
    for s in range(0, len(X), local_cfg.eval_batch_size):
        xb = preprocess_for_resnet(X[s:s+local_cfg.eval_batch_size].to(device), local_cfg.model_image_size)
        logits, q, ctx = model(xb, t=0.0, return_measurements=True)
        q_rows.append(q.detach().cpu().numpy())
        ctx_rows.append(ctx.detach().cpu().numpy())
        prob_rows.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
    return np.concatenate(q_rows), np.concatenate(ctx_rows), np.concatenate(prob_rows)

@torch.no_grad()
def extract_classical_context(model, X, local_cfg, device):
    if model is None:
        return None
    model.eval(); rows = []
    for s in range(0, len(X), local_cfg.eval_batch_size):
        xb = preprocess_for_resnet(X[s:s+local_cfg.eval_batch_size].to(device), local_cfg.model_image_size)
        _, ctx = model(xb, return_context=True)
        rows.append(ctx.detach().cpu().numpy())
    return np.concatenate(rows)

def low_frequency_suppress(X, radius=10):
    X_np = X.detach().cpu().numpy().astype(np.float32)
    out = np.zeros_like(X_np)
    for i in range(X_np.shape[0]):
        for c in range(X_np.shape[1]):
            Freq = np.fft.fft2(X_np[i, c])
            amp, phase = np.abs(Freq), np.angle(Freq)
            amp_shift = np.fft.fftshift(amp)
            H, W = amp.shape; cy, cx = H//2, W//2
            yy, xx = np.ogrid[:H, :W]
            mask = (yy-cy)**2 + (xx-cx)**2 <= radius**2
            amp_shift[mask] = np.median(amp_shift)
            amp_new = np.fft.ifftshift(amp_shift)
            out[i, c] = np.real(np.fft.ifft2(amp_new * np.exp(1j*phase)))
    return torch.tensor(np.clip(out, 0, 1), dtype=X.dtype)

def benign_augment_for_blend(X):
    # deterministic, mild augmentation for instability probe
    X_flip = torch.flip(X, dims=[3])
    X_noise = (X + 0.015 * torch.randn_like(X)).clamp(0, 1)
    return (X_flip + X_noise) / 2.0

def extract_qmrs_probe_features(qnn_model, X_pool, local_cfg, device, attack_name):
    q0, ctx, probs0 = extract_qnn_features(qnn_model, X_pool, local_cfg, device)
    if attack_name == "blend":
        X_probe = benign_augment_for_blend(X_pool)
    else:
        X_probe = low_frequency_suppress(X_pool, radius=local_cfg.fiba_low_freq_radius)
    q_probe, _, probs_probe = extract_qnn_features(qnn_model, X_probe, local_cfg, device)
    measurement_delta = np.linalg.norm(q0 - q_probe, axis=1)
    target_conf = probs0[:, local_cfg.target_class]
    target_conf_delta = probs0[:, local_cfg.target_class] - probs_probe[:, local_cfg.target_class]
    return np.concatenate([q0, ctx, measurement_delta[:, None], target_conf[:, None], target_conf_delta[:, None]], axis=1)

def detector_scores_from_features(X_val_feat, y_val, X_test_feat, local_cfg, k):
    scaler = StandardScaler()
    Vv0 = scaler.fit_transform(X_val_feat)
    Vt0 = scaler.transform(X_test_feat)
    n_comp = min(local_cfg.ica_components, Vv0.shape[1], max(1, Vv0.shape[0]-1))
    reducer = FastICA(n_components=n_comp, random_state=local_cfg.primary_seed, whiten="unit-variance", max_iter=1000, tol=1e-3)
    try:
        Vv = reducer.fit_transform(Vv0); Vt = reducer.transform(Vt0)
    except Exception:
        reducer = PCA(n_components=n_comp, random_state=local_cfg.primary_seed)
        Vv = reducer.fit_transform(Vv0); Vt = reducer.transform(Vt0)
    km = KMeans(n_clusters=k, random_state=local_cfg.primary_seed, n_init=local_cfg.kmeans_n_init)
    val_cluster = km.fit_predict(Vv)
    centers = km.cluster_centers_
    # Select suspicious cluster on validation only: highest poison proportion, tie smaller size.
    rows = []
    for c in range(k):
        mask = val_cluster == c
        rows.append((c, y_val[mask].mean() if mask.any() else 0.0, mask.sum()))
    suspicious_cluster = sorted(rows, key=lambda x: (-x[1], x[2]))[0][0]
    def score(V):
        d_susp = np.linalg.norm(V - centers[suspicious_cluster], axis=1)
        other = [j for j in range(k) if j != suspicious_cluster]
        d_other = np.min(np.stack([np.linalg.norm(V - centers[j], axis=1) for j in other], axis=1), axis=1)
        return d_other - d_susp
    return score(Vv), score(Vt), {"k": k, "suspicious_cluster": int(suspicious_cluster), "cluster_rows": rows}

def prediction_from_scores(val_scores, y_val, test_scores, expected_poison_count, method="clean_val_95pct", fpr=0.05):
    if method == "clean_val_95pct":
        clean_val_scores = val_scores[np.asarray(y_val) == 0]
        tau = float(np.percentile(clean_val_scores, 100*(1-fpr)))
        return (test_scores >= tau).astype(int), tau, "clean_val_95pct"
    pred = np.zeros(len(test_scores), dtype=int)
    top = np.argsort(test_scores)[-int(expected_poison_count):]
    pred[top] = 1
    return pred, None, "top_expected_poison_count"

def eval_detection(y_true, scores, pred):
    out = {
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "AUPRC": float(average_precision_score(y_true, scores)),
    }
    try:
        out["AUROC"] = float(roc_auc_score(y_true, scores))
    except Exception:
        out["AUROC"] = np.nan
    return out

def select_k_and_evaluate(feature_name, X_val_feat, y_val, X_test_feat, y_test, local_cfg, expected_poison_count):
    val_rows = []
    candidates = {}
    for k in local_cfg.k_values:
        try:
            val_scores, test_scores, meta = detector_scores_from_features(X_val_feat, y_val, X_test_feat, local_cfg, k)
            val_pred, val_tau, _ = prediction_from_scores(val_scores, y_val, val_scores, expected_poison_count=max(1, int(y_val.sum())), method=local_cfg.threshold_method, fpr=local_cfg.fpr_target)
            m_val = eval_detection(y_val, val_scores, val_pred)
            val_rows.append({"Feature Space": feature_name, "K": k, **m_val})
            candidates[k] = (val_scores, test_scores, meta)
        except Exception as e:
            val_rows.append({"Feature Space": feature_name, "K": k, "error": str(e)})
    val_df = pd.DataFrame(val_rows)
    valid = val_df.dropna(subset=["AUPRC", "F1"], how="any")
    if valid.empty:
        raise RuntimeError(f"No valid K for {feature_name}. Details: {val_df}")
    best_k = int(valid.sort_values(["AUPRC", "F1"], ascending=False).iloc[0]["K"])
    val_scores, test_scores, meta = candidates[best_k]
    test_pred, tau, threshold_name = prediction_from_scores(val_scores, y_val, test_scores, expected_poison_count=expected_poison_count, method=local_cfg.threshold_method, fpr=local_cfg.fpr_target)
    test_metrics = eval_detection(y_test, test_scores, test_pred)
    row = {"Feature Space": feature_name, "K": best_k, "Threshold": threshold_name, "Tau": tau, **test_metrics}
    return row, val_df


In [5]:

# ============================================================
# Attack-success-first runner for Blend and FIBA
# Test set is used once after validation pair/K/threshold choices are frozen.
# ============================================================

def run_attack_extension(dataset_name, attack_name, base_cfg, seed):
    assert attack_name in ["blend", "fiba"]
    assert dataset_name in ["dermamnist", "pathmnist"]
    key = f"{dataset_name}_{'blend' if attack_name=='blend' else 'fiba'}"
    print("\n" + "="*90)
    print(f"Running {attack_name.upper()} extension on {dataset_name}")
    print("="*90)

    bundle = load_dataset_bundle(dataset_name)
    attack_values = base_cfg.blend_alpha_values if attack_name == "blend" else base_cfg.fiba_strength_values
    pair_candidates = CANDIDATE_PAIRS[key]

    pilot_rows = []
    selected = None
    if RUN_PAIR_SELECTION:
        print("Validation-only pair/strength selection")
        for source, target in pair_candidates:
            for val in attack_values:
                local_cfg = clone_cfg(
                    base_cfg,
                    dataset_name=dataset_name, attack_type=attack_name, n_classes=bundle["n_classes"],
                    source_class=source, target_class=target,
                    max_train_n=base_cfg.pilot_max_train_n, clean_epochs=base_cfg.pilot_epochs,
                    blend_alpha=val if attack_name == "blend" else base_cfg.blend_alpha,
                    fiba_strength=val if attack_name == "fiba" else base_cfg.fiba_strength,
                )
                try:
                    art = train_poisoned_qnn_and_classical(
                        bundle, local_cfg, attack_name, seed,
                        max_train_n=local_cfg.max_train_n,
                        epochs_override=local_cfg.clean_epochs,
                        eval_split="val",
                        train_classical_baseline=False,
                    )
                    row = {
                        "Attack": attack_name, "Dataset": dataset_name, "Source": source, "Target": target,
                        "Strength/Alpha": val, "Poison Rate": local_cfg.blend_poison_rate if attack_name=='blend' else local_cfg.fiba_poison_rate,
                        "Eval Split": art.get("eval_split", "val"), "Classical Trained?": art.get("classical_trained", False), "Val/Pilot CA": art["qnn_ca"]["accuracy"], "Val/Pilot ASR": art["qnn_asr"]["accuracy"],
                        "Gate Passed?": bool(art["qnn_asr"]["accuracy"] >= local_cfg.asr_min and art["qnn_ca"]["accuracy"] >= local_cfg.clean_acc_min),
                    }
                except Exception as e:
                    row = {"Attack": attack_name, "Dataset": dataset_name, "Source": source, "Target": target, "Strength/Alpha": val, "error": str(e), "Gate Passed?": False}
                pilot_rows.append(row)
        pilot_df = pd.DataFrame(pilot_rows)
        display(pilot_df)
        pilot_path = os.path.join(base_cfg.out_dir, f"{dataset_name}_{attack_name}_validation_pair_selection.csv")
        pilot_df.to_csv(pilot_path, index=False)
        passed = pilot_df[pilot_df["Gate Passed?"] == True].copy()
        if passed.empty:
            status = "blocked_by_weak_attack"
            msg = "No valid source-target pair reached ASR/CA gate. Increase attack strength, poison rate, or training budget."
            print("[BLOCKED]", msg)
            return {"status": status, "attack_success_df": pilot_df, "detection_df": pd.DataFrame(), "message": msg}
        selected = passed.sort_values(["Val/Pilot ASR", "Val/Pilot CA"], ascending=False).iloc[0]
        source, target, selected_value = int(selected["Source"]), int(selected["Target"]), float(selected["Strength/Alpha"])
    else:
        source, target = pair_candidates[0]
        selected_value = attack_values[0]

    # Final training with frozen pair/strength selected only from validation/pilot.
    final_cfg = clone_cfg(
        base_cfg,
        dataset_name=dataset_name, attack_type=attack_name, n_classes=bundle["n_classes"],
        source_class=source, target_class=target,
        max_train_n=5000 if attack_name == "blend" else 6000,
        clean_epochs=8,
        blend_alpha=selected_value if attack_name == "blend" else base_cfg.blend_alpha,
        fiba_strength=selected_value if attack_name == "fiba" else base_cfg.fiba_strength,
    )
    final_art = train_poisoned_qnn_and_classical(
        bundle, final_cfg, attack_name, seed,
        max_train_n=final_cfg.max_train_n,
        epochs_override=final_cfg.clean_epochs,
        eval_split="test",
        train_classical_baseline=RUN_CLASSICAL_POISONED_BASELINE,
    )
    qnn_ca, qnn_asr = final_art["qnn_ca"]["accuracy"], final_art["qnn_asr"]["accuracy"]
    classical_ca = final_art["classical_ca"]["accuracy"] if final_art["classical_ca"] else np.nan
    classical_asr = final_art["classical_asr"]["accuracy"] if final_art["classical_asr"] else np.nan
    gate_passed = bool(qnn_asr >= final_cfg.asr_min)

    attack_success_df = pd.DataFrame([{
        "Attack": attack_name, "Dataset": dataset_name, "Source→Target": f"{source}->{target}",
        "Strength/Alpha": selected_value, "Poison Rate": final_cfg.blend_poison_rate if attack_name=='blend' else final_cfg.fiba_poison_rate,
        "Eval Split": final_art.get("eval_split", "test"), "Classical Trained?": final_art.get("classical_trained", False), "QNN CA": qnn_ca, "QNN ASR": qnn_asr,
        "Classical Poisoned CA": classical_ca, "Classical Poisoned ASR": classical_asr,
        "Gate Passed?": gate_passed,
        "Status": "passed_attack_gate" if gate_passed else "blocked_by_weak_attack",
    }])
    display(attack_success_df)
    success_path = os.path.join(base_cfg.out_dir, f"{dataset_name}_{attack_name}_attack_success.csv")
    attack_success_df.to_csv(success_path, index=False)

    if not gate_passed:
        blocked_df = pd.DataFrame([{
            "Attack": attack_name, "Dataset": dataset_name, "Status": "blocked_by_weak_attack",
            "Reason": "ASR < 0.80; detection is not meaningful until the poisoned QNN learns the trigger.",
        }])
        det_path = os.path.join(base_cfg.out_dir, f"{dataset_name}_{attack_name}_detection_blocked.csv")
        blocked_df.to_csv(det_path, index=False)
        print("[BLOCKED] Detection not run because ASR gate failed. Saved:", det_path)
        return {"status": "blocked_by_weak_attack", "attack_success_df": attack_success_df, "detection_df": blocked_df}

    # Build validation and test pools after final pair/strength is frozen.
    X_val_pool, y_val_poison, val_groups, val_summary = build_qsentry_poison_pool(bundle["X_val"], bundle["y_val"], final_art["trigger"], final_cfg, attack_name, seed)
    X_test_pool, y_test_poison, test_groups, test_summary = build_qsentry_poison_pool(bundle["X_test"], bundle["y_test"], final_art["trigger"], final_cfg, attack_name, seed+17)

    # Extract features.
    q_val, qctx_val, _ = extract_qnn_features(final_art["qnn"], X_val_pool, final_cfg, DEVICE)
    q_test, qctx_test, _ = extract_qnn_features(final_art["qnn"], X_test_pool, final_cfg, DEVICE)
    raw_val = X_val_pool.reshape(len(X_val_pool), -1).numpy().astype(np.float32)
    raw_test = X_test_pool.reshape(len(X_test_pool), -1).numpy().astype(np.float32)
    classical_ctx_val = extract_classical_context(final_art["classical"], X_val_pool, final_cfg, DEVICE)
    classical_ctx_test = extract_classical_context(final_art["classical"], X_test_pool, final_cfg, DEVICE)
    qmrs_val = extract_qmrs_probe_features(final_art["qnn"], X_val_pool, final_cfg, DEVICE, attack_name)
    qmrs_test = extract_qmrs_probe_features(final_art["qnn"], X_test_pool, final_cfg, DEVICE, attack_name)

    feature_spaces = {
        "Raw pixel baseline": (raw_val, raw_test),
        "QNN context baseline": (qctx_val, qctx_test),
        "Quantum measurement-only": (q_val, q_test),
        "CC-QMC-inspired measurement": (q_val, q_test),
        "QMRS attack-probe": (qmrs_val, qmrs_test),
    }
    if classical_ctx_val is not None:
        feature_spaces["Classical context baseline"] = (classical_ctx_val, classical_ctx_test)

    det_rows, val_k_rows = [], []
    expected_poison_count = int(y_test_poison.sum())
    for name, (fv, ft) in feature_spaces.items():
        row, val_df = select_k_and_evaluate(name, fv, y_val_poison, ft, y_test_poison, final_cfg, expected_poison_count)
        row.update({"Attack": attack_name, "Dataset": dataset_name, "CA": qnn_ca, "ASR": qnn_asr})
        det_rows.append(row)
        val_df["Attack"] = attack_name; val_df["Dataset"] = dataset_name
        val_k_rows.append(val_df)

    detection_df = pd.DataFrame(det_rows)[["Attack", "Dataset", "Feature Space", "K", "Threshold", "CA", "ASR", "F1", "Precision", "Recall", "AUPRC", "AUROC"]]
    validation_k_df = pd.concat(val_k_rows, ignore_index=True)
    display(detection_df.sort_values("AUPRC", ascending=False))
    det_path = os.path.join(base_cfg.out_dir, f"{dataset_name}_{attack_name}_detection_results.csv")
    kval_path = os.path.join(base_cfg.out_dir, f"{dataset_name}_{attack_name}_validation_k_selection.csv")
    detection_df.to_csv(det_path, index=False)
    validation_k_df.to_csv(kval_path, index=False)

    # Sanitization is deliberately not claimed unless executed.
    sanitization_df = pd.DataFrame([{
        "Attack": attack_name, "Dataset": dataset_name, "CA Before": qnn_ca, "ASR Before": qnn_asr,
        "Samples Flagged": "planned / pending experiment",
        "Samples Removed/Sanitized": "planned / pending experiment",
        "CA After": "planned / pending experiment",
        "ASR After": "planned / pending experiment",
        "ASR Reduction": "planned / pending experiment",
    }])
    san_path = os.path.join(base_cfg.out_dir, f"{dataset_name}_{attack_name}_sanitization_pending.csv")
    sanitization_df.to_csv(san_path, index=False)
    return {"status": "completed", "attack_success_df": attack_success_df, "detection_df": detection_df, "sanitization_df": sanitization_df}

all_attack_success, all_detection, all_sanitization = [], [], []

if RUN_DERMAMNIST_BLEND:
    blend_out = run_attack_extension("dermamnist", "blend", cfg, cfg.primary_seed)
    all_attack_success.append(blend_out.get("attack_success_df", pd.DataFrame()))
    all_detection.append(blend_out.get("detection_df", pd.DataFrame()))
    all_sanitization.append(blend_out.get("sanitization_df", pd.DataFrame()))
else:
    PLANNED_PENDING.append("DermaMNIST Blend planned / pending experiment")

if RUN_PATHMNIST_FIBA:
    fiba_cfg = clone_cfg(cfg, dataset_name="pathmnist", attack_type="fiba")
    fiba_out = run_attack_extension("pathmnist", "fiba", fiba_cfg, cfg.primary_seed)
    all_attack_success.append(fiba_out.get("attack_success_df", pd.DataFrame()))
    all_detection.append(fiba_out.get("detection_df", pd.DataFrame()))
    all_sanitization.append(fiba_out.get("sanitization_df", pd.DataFrame()))
else:
    print("FIBA disabled by default. Enable RUN_PATHMNIST_FIBA only after Blend is validated.")
    PLANNED_PENDING.append("PathMNIST FIBA planned / pending experiment")

attack_success_final = pd.concat([d for d in all_attack_success if d is not None and not d.empty], ignore_index=True) if all_attack_success else pd.DataFrame()
detection_final = pd.concat([d for d in all_detection if d is not None and not d.empty], ignore_index=True) if all_detection else pd.DataFrame()
sanitization_final = pd.concat([d for d in all_sanitization if d is not None and not d.empty], ignore_index=True) if all_sanitization else pd.DataFrame()

# Required final tables.
attack_success_path = os.path.join(cfg.out_dir, "blend_fiba_attack_success_table.csv")
detection_path = os.path.join(cfg.out_dir, "blend_fiba_detection_table.csv")
sanitization_path = os.path.join(cfg.out_dir, "blend_fiba_sanitization_table.csv")
report_path = os.path.join(cfg.out_dir, "blend_fiba_run_report.json")
attack_success_final.to_csv(attack_success_path, index=False)
detection_final.to_csv(detection_path, index=False)
sanitization_final.to_csv(sanitization_path, index=False)

run_report = {
    "scope": "Blend/DermaMNIST and FIBA/PathMNIST poisonous-image detection extension",
    "default_policy": "Blend runs first; FIBA disabled by default until Blend is validated.",
    "qsentry_claim_guard": "Use QSentry-style / CC-QMC-inspired wording unless exactly reproducing QSentry.",
    "config": asdict(cfg),
    "planned_pending": PLANNED_PENDING,
    "needs_verification": sorted(set(NEEDS_VERIFICATION)),
    "outputs": {
        "attack_success": attack_success_path,
        "detection": detection_path,
        "sanitization": sanitization_path,
    },
}
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(run_report, f, indent=2)

print("\nFinal attack-success table:")
display(attack_success_final if not attack_success_final.empty else pd.DataFrame([{"Status":"planned / pending experiment"}]))
print("\nFinal detection table:")
display(detection_final if not detection_final.empty else pd.DataFrame([{"Status":"planned / pending experiment"}]))
print("\nFinal sanitization table:")
display(sanitization_final if not sanitization_final.empty else pd.DataFrame([{"Status":"planned / pending experiment"}]))
print("\nSaved:", attack_success_path, detection_path, sanitization_path, report_path, sep="\n- ")



Running BLEND extension on dermamnist


100%|██████████| 19.7M/19.7M [00:01<00:00, 12.7MB/s]


Loaded dermamnist: torch.Size([7007, 3, 28, 28]) torch.Size([1003, 3, 28, 28]) torch.Size([2005, 3, 28, 28])


,class_id,class_name
0,0,actinic keratoses and intraepithelial carcinoma
1,1,basal cell carcinoma
2,2,benign keratosis-like lesions
3,3,dermatofibroma
4,4,melanoma
5,5,melanocytic nevi
6,6,vascular lesions


Validation-only pair/strength selection
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


...


Final detection table:


,Attack,Dataset,Feature Space,K,Threshold,CA,ASR,F1,Precision,Recall,AUPRC,AUROC
0,blend,dermamnist,Raw pixel baseline,8,clean_val_95pct,0.72,0.995,0.147368,0.155556,0.14,0.100489,0.573581
1,blend,dermamnist,QNN context baseline,8,clean_val_95pct,0.72,0.995,0.781250,0.641026,1.00,1.000000,1.000000
2,blend,dermamnist,Quantum measurement-only,8,clean_val_95pct,0.72,0.995,0.645161,0.476190,1.00,0.334479,0.934086
3,blend,dermamnist,CC-QMC-inspired measurement,8,clean_val_95pct,0.72,0.995,0.645161,0.476190,1.00,0.334479,0.934086
4,blend,dermamnist,QMRS attack-probe,10,clean_val_95pct,0.72,0.995,0.800000,0.666667,1.00,1.000000,1.000000
5,blend,dermamnist,Classical context baseline,5,clean_val_95pct,0.72,0.995,0.735294,0.581395,1.00,0.948479,0.997236



Final sanitization table:


,Attack,Dataset,CA Before,ASR Before,Samples Flagged,Samples Removed/Sanitized,CA After,ASR After,ASR Reduction
0,blend,dermamnist,0.72,0.995,planned / pending experiment,planned / pending experiment,planned / pending experiment,planned / pending experiment,planned / pending experiment



Saved:
- ./blend_fiba_qsentry_extension_outputs/blend_fiba_attack_success_table.csv
- ./blend_fiba_qsentry_extension_outputs/blend_fiba_detection_table.csv
- ./blend_fiba_qsentry_extension_outputs/blend_fiba_sanitization_table.csv
- ./blend_fiba_qsentry_extension_outputs/blend_fiba_run_report.json
